In [0]:
from pyspark.sql import functions as F

orders = spark.table("workspace.default.silver_orders_priced")
payments = spark.table("workspace.default.silver_payments")

orders_payments = orders.join(payments, on="order_id", how="left")

unpaid_orders = (orders_payments
    .filter(F.col("payment_amount").isNull())
    .withColumn("issue_type", F.lit("UNPAID_ORDER"))
)
print("Unpaid orders:", unpaid_orders.count())
unpaid_orders.display()

Unpaid orders: 2037


order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount,payment_amount,payment_method,payment_date,issue_type
8,100,490,5,823.93,369.61,2024-02-29,Cancelled,Web,0.06,null,null,null,UNPAID_ORDER
23,15,396,3,1087.24,859.85,2024-02-24,Completed,Web,0.03,null,null,null,UNPAID_ORDER
32,51,1058,2,136.91,1795.45,2024-01-10,Completed,Web,0.11,null,null,null,UNPAID_ORDER
46,84,1981,2,1922.04,456.82,2024-02-09,Completed,Store,0.01,null,null,null,UNPAID_ORDER
55,21,1394,4,324.28,500.9,2024-01-15,Cancelled,Store,0.03,null,null,null,UNPAID_ORDER
57,23,22,2,1603.94,339.97,2024-02-18,Cancelled,Web,0.11,null,null,null,UNPAID_ORDER
60,80,1833,1,1312.15,1059.51,2024-02-14,Completed,App,0.08,null,null,null,UNPAID_ORDER
68,68,534,2,163.37,533.94,2024-02-13,Completed,Store,0.2,null,null,null,UNPAID_ORDER
73,67,321,1,1100.87,661.68,2024-01-30,Completed,App,0.18,null,null,null,UNPAID_ORDER
78,46,1912,5,294.82,1761.64,2024-02-15,Cancelled,App,0.12,null,null,null,UNPAID_ORDER


In [0]:
orphan_payments = (payments.join(orders, on="order_id", how="left_anti")
    .withColumn("issue_type", F.lit("ORPHAN_PAYMENT"))
)
print("Orphan payments:", orphan_payments.count())
orphan_payments.display()

Orphan payments: 600


order_id,payment_amount,payment_method,payment_date,issue_type
20001,1794.72,Card,2024-01-15,ORPHAN_PAYMENT
20002,940.73,UPI,2024-01-15,ORPHAN_PAYMENT
20003,599.08,Card,2024-01-15,ORPHAN_PAYMENT
20004,1440.16,Wallet,2024-01-15,ORPHAN_PAYMENT
20005,1386.48,Card,2024-01-15,ORPHAN_PAYMENT
20006,1422.23,Wallet,2024-01-15,ORPHAN_PAYMENT
20007,1773.22,UPI,2024-01-15,ORPHAN_PAYMENT
20008,1223.31,Wallet,2024-01-15,ORPHAN_PAYMENT
20009,547.5,Wallet,2024-01-15,ORPHAN_PAYMENT
20010,625.97,NetBanking,2024-01-15,ORPHAN_PAYMENT


In [0]:
order_counts = orders.groupBy("order_id").count()

duplicate_order_ids = order_counts.filter(F.col("count") > 1).select("order_id")

duplicate_orders = (orders.join(duplicate_order_ids, on="order_id", how="inner")
    .withColumn("issue_type", F.lit("DUPLICATE_ORDER"))
)
print("Duplicate order rows:", duplicate_orders.count())
duplicate_orders.display()

Duplicate order rows: 800


order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount,issue_type
6,31,741,3,625.85,1678.43,2024-01-31,Completed,Store,0.15,DUPLICATE_ORDER
21,18,813,2,1238.8,926.46,2024-02-07,Cancelled,Web,0.01,DUPLICATE_ORDER
34,86,1686,4,1649.51,1738.61,2024-02-05,Cancelled,Web,0.03,DUPLICATE_ORDER
56,82,891,5,332.01,1296.03,2024-01-17,Completed,Web,0.28,DUPLICATE_ORDER
93,58,57,5,477.31,2099.07,2024-02-09,Completed,App,0.25,DUPLICATE_ORDER
240,40,1426,2,875.1,325.3,2024-01-20,Completed,App,0.07,DUPLICATE_ORDER
351,48,1427,3,1537.07,1661.19,2024-02-02,Completed,Web,0.13,DUPLICATE_ORDER
399,72,617,4,1020.54,285.57,2024-01-05,Completed,Web,0.27,DUPLICATE_ORDER
429,43,1713,1,443.11,737.38,2024-02-18,Cancelled,Store,0.15,DUPLICATE_ORDER
716,57,312,3,1693.25,688.35,2024-02-01,Cancelled,Store,0.27,DUPLICATE_ORDER


In [0]:
price_mismatch = (orders
    .filter(F.col("order_price") != F.col("catalog_price"))
    .withColumn("issue_type", F.lit("PRICE_MISMATCH"))
    .withColumn("price_diff", F.col("order_price") - F.col("catalog_price"))
)
print("Price mismatches:", price_mismatch.count())
price_mismatch.display()

Price mismatches: 20400


order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount,issue_type,price_diff
1,29,1950,4,1701.72,1136.85,2024-02-29,Cancelled,Web,0.05,PRICE_MISMATCH,564.8700000000001
2,98,1491,5,1643.01,1658.33,2024-01-28,Completed,Store,0.16,PRICE_MISMATCH,-15.319999999999936
3,2,1518,4,1948.47,1837.12,2024-01-10,Completed,Store,0.18,PRICE_MISMATCH,111.35000000000014
4,33,440,3,1911.69,1071.5,2024-02-01,Completed,Web,0.07,PRICE_MISMATCH,840.19
5,1,1011,4,795.88,1781.59,2024-02-11,Completed,App,0.25,PRICE_MISMATCH,-985.7099999999999
6,31,741,3,625.85,1678.43,2024-01-31,Completed,Store,0.15,PRICE_MISMATCH,-1052.58
7,35,201,3,174.15,1507.39,2024-01-14,Completed,Web,0.22,PRICE_MISMATCH,-1333.24
8,100,490,5,823.93,369.61,2024-02-29,Cancelled,Web,0.06,PRICE_MISMATCH,454.31999999999994
9,39,269,1,1880.09,278.64,2024-02-10,Cancelled,Web,0.27,PRICE_MISMATCH,1601.4499999999998
10,50,1203,3,522.32,570.28,2024-01-12,Completed,App,0.19,PRICE_MISMATCH,-47.95999999999992


In [0]:
from pyspark.sql import functions as F

diag = (orders
    .withColumn("pct_diff", F.round(F.abs(F.col("order_price") - F.col("catalog_price")) / F.col("catalog_price") * 100, 1))
)

diag.groupBy(
    F.when(F.col("pct_diff") <= 5, "0-5% (likely fine)")
     .when(F.col("pct_diff") <= 30, "5-30% (could be discount)")
     .otherwise("30%+ (likely real anomaly)")
     .alias("bucket")
).count().display()

bucket,count
30%+ (likely real anomaly),14689
0-5% (likely fine),987
5-30% (could be discount),4724


In [0]:
print("Total rows in orders:", orders.count())
print("Distinct order_ids:", orders.select("order_id").distinct().count())

Total rows in orders: 20400
Distinct order_ids: 20000


In [0]:
bronze = spark.table("workspace.default.bronze_orders")
print("Bronze total rows:", bronze.count())
print("Bronze distinct order_ids:", bronze.select("order_id").distinct().count())

Bronze total rows: 20400
Bronze distinct order_ids: 20000


In [0]:
raw_check = spark.read.option("header", True).csv("/Volumes/workspace/default/shadow_revenue_data/orders.csv")
print("Raw CSV row count:", raw_check.count())

Raw CSV row count: 20400


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/shadow_revenue_data/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/shadow_revenue_data/customers.csv,customers.csv,63897,1783662895000
dbfs:/Volumes/workspace/default/shadow_revenue_data/orders.csv,orders.csv,1077015,1783662912000
dbfs:/Volumes/workspace/default/shadow_revenue_data/payments.csv,payments.csv,569799,1783662908000
dbfs:/Volumes/workspace/default/shadow_revenue_data/products.csv,products.csv,7588,1783662895000


In [0]:
# Look at all price history for one product
spark.table("workspace.default.silver_products").filter(F.col("product_id") == 29).orderBy("effective_date").display()

product_id,price,category,effective_date,end_date,is_current
29,1209.98,Electronics,2023-01-01,2023-12-31,0
29,1136.85,Electronics,2024-01-01,null,1


In [0]:
# Look at the actual order row for order_id 1 (product_id 29, order_date visible in your earlier screenshot)
spark.table("workspace.default.silver_orders_priced").filter(F.col("order_id") == 1).display()

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,1136.85,2024-02-29,Cancelled,Web,0.05


In [0]:
price_mismatch = (orders
    .withColumn("price_diff", F.round(F.col("order_price") - F.col("catalog_price"), 2))
    .withColumn("pct_diff", F.round(F.abs(F.col("price_diff")) / F.col("catalog_price") * 100, 1))
    .filter(F.col("pct_diff") > 10)
    .withColumn("issue_type", F.lit("PRICE_MISMATCH"))
)
print("Price mismatches:", price_mismatch.count())

Price mismatches: 18475


In [0]:
gold_shadow_revenue = (
    unpaid_orders.select("order_id", "order_price", "issue_type")
    .unionByName(orphan_payments.select("order_id", F.col("payment_amount").alias("order_price"), "issue_type"), allowMissingColumns=True)
    .unionByName(duplicate_orders.select("order_id", "order_price", "issue_type"), allowMissingColumns=True)
    .unionByName(price_mismatch.select("order_id", "order_price", "issue_type"), allowMissingColumns=True)
)
gold_shadow_revenue.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_shadow_revenue")

report = (gold_shadow_revenue.groupBy("issue_type")
    .agg(F.count("*").alias("num_incidents"), F.sum("order_price").alias("total_leaked_revenue"))
    .orderBy(F.desc("total_leaked_revenue")))
report.display()

issue_type,num_incidents,total_leaked_revenue
PRICE_MISMATCH,18475,1.8869746390000038E7
UNPAID_ORDER,2037,2065247.489999999
DUPLICATE_ORDER,800,831077.4399999996
ORPHAN_PAYMENT,600,619675.120000001


In [0]:
price_mismatch = (orders
    .withColumn("price_diff", F.round(F.col("order_price") - F.col("catalog_price"), 2))
    .withColumn("pct_diff", F.round(F.abs(F.col("price_diff")) / F.col("catalog_price") * 100, 1))
    .filter(F.col("pct_diff") > 25)
    .withColumn("issue_type", F.lit("PRICE_MISMATCH"))
)
print("Price mismatches:", price_mismatch.count())

Price mismatches: 15567


In [0]:
gold_shadow_revenue = (
    unpaid_orders.select("order_id", "order_price", "issue_type")
    .unionByName(orphan_payments.select("order_id", F.col("payment_amount").alias("order_price"), "issue_type"), allowMissingColumns=True)
    .unionByName(duplicate_orders.select("order_id", "order_price", "issue_type"), allowMissingColumns=True)
    .unionByName(price_mismatch.select("order_id", "order_price", "issue_type"), allowMissingColumns=True)
)
gold_shadow_revenue.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_shadow_revenue")

report = (gold_shadow_revenue.groupBy("issue_type")
    .agg(F.count("*").alias("num_incidents"), F.sum("order_price").alias("total_leaked_revenue"))
    .orderBy(F.desc("total_leaked_revenue")))
report.display()

issue_type,num_incidents,total_leaked_revenue
PRICE_MISMATCH,15567,1.5383404300000027E7
UNPAID_ORDER,2037,2065247.489999999
DUPLICATE_ORDER,800,831077.4399999996
ORPHAN_PAYMENT,600,619675.120000001
